# 05 — Kafka Streaming, Offsets, Replay, and Idempotent Sinks

## Business scenario

Checkout services publish order-status events continuously. A consumer can crash
after writing output but before committing its offset, so a record may be delivered
again. The sink must remain correct under at-least-once delivery.

This notebook runs in deterministic simulation mode by default. That mode teaches
the state transitions without Docker. The optional section uses the included real
Apache Kafka container. Do not describe simulation mode as Kafka broker experience.

### Learning objectives

- Explain topics, keys, partitions, consumer groups, and offsets.
- Observe how a key provides stable partition affinity.
- Separate processing from offset commit.
- Replay a batch and keep an idempotent sink correct.
- Distinguish event time from processing time and identify late events.
- Run the same ideas against a real local Kafka broker.


In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from hashlib import sha256
from pathlib import Path
from typing import Any
import json
import random
import time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
STREAM_ROOT = PROJECT_ROOT / "lab_data" / "streaming"
STREAM_ROOT.mkdir(parents=True, exist_ok=True)


## A small simulator with Kafka-like state

The simulator is intentionally limited; it is not a broker. It makes three pieces
of state visible:

- an append-only log for each partition;
- the next offset for each consumer group and partition;
- a sink keyed by unique `event_id`.

We use SHA-256 rather than Python's built-in `hash()` so key-to-partition mapping is
stable across interpreter restarts.


In [2]:
@dataclass(frozen=True)
class Message:
    topic: str
    partition: int
    offset: int
    key: str
    value: dict[str, Any]


class LocalKafkaSimulator:
    def __init__(self, topic: str, partitions: int = 3):
        self.topic = topic
        self.logs: list[list[Message]] = [[] for _ in range(partitions)]
        self.group_offsets: dict[tuple[str, int], int] = {}

    def _partition_for(self, key: str) -> int:
        return int(sha256(key.encode()).hexdigest(), 16) % len(self.logs)

    def produce(self, key: str, value: dict[str, Any]) -> Message:
        partition = self._partition_for(key)
        message = Message(self.topic, partition, len(self.logs[partition]), key, value)
        self.logs[partition].append(message)
        return message

    def poll(self, group_id: str, max_records: int = 10) -> list[Message]:
        result = []
        for partition, log in enumerate(self.logs):
            next_offset = self.group_offsets.get((group_id, partition), 0)
            for message in log[next_offset:]:
                result.append(message)
                if len(result) == max_records:
                    return result
        return result

    def commit(self, group_id: str, messages: list[Message]) -> None:
        for message in messages:
            key = (group_id, message.partition)
            self.group_offsets[key] = max(self.group_offsets.get(key, 0), message.offset + 1)

    def reset_group(self, group_id: str) -> None:
        for partition in range(len(self.logs)):
            self.group_offsets[(group_id, partition)] = 0


broker = LocalKafkaSimulator("order-status", partitions=3)


In [3]:
rng = random.Random(41)
start = datetime(2026, 1, 5, tzinfo=timezone.utc)
source_events = []
for index in range(45):
    event_time = start + timedelta(seconds=index * 20)
    if index == 40:
        event_time = start + timedelta(minutes=1)  # late event arriving near the end
    event = {
        "event_id": f"status-{index:04d}",
        "order_id": f"ord_{rng.randint(1, 12):06d}",
        "status": rng.choice(["paid", "packed", "shipped"]),
        "event_time": event_time.isoformat(),
        "produced_at": datetime.now(timezone.utc).isoformat(),
    }
    source_events.append(event)

# Produce every event and repeat two to simulate at-least-once publication.
for event in source_events + [source_events[8], source_events[17]]:
    broker.produce(key=event["order_id"], value=event)

partition_sizes = [len(log) for log in broker.logs]
partition_sizes, sum(partition_sizes)


([8, 14, 25], 47)

## Processing and commit are separate failure points

The consumer writes records to an idempotent sink keyed by `event_id`. Imagine it
crashes after the write but before `commit()`. The next poll returns those messages
again. Correctness must not depend on exactly-once delivery from the network.


In [4]:
GROUP_ID = "silver-order-status-v1"
sink: dict[str, dict] = {}


def write_idempotently(messages: list[Message]) -> dict:
    inserted = 0
    duplicates = 0
    for message in messages:
        event_id = message.value["event_id"]
        if event_id in sink:
            duplicates += 1
        else:
            sink[event_id] = {
                **message.value,
                "kafka_partition": message.partition,
                "kafka_offset": message.offset,
            }
            inserted += 1
    return {"inserted": inserted, "duplicates": duplicates}


first_batch = broker.poll(GROUP_ID, max_records=12)
first_write = write_idempotently(first_batch)
# Deliberate crash: do not commit.
replayed_batch = broker.poll(GROUP_ID, max_records=12)
replay_write = write_idempotently(replayed_batch)
broker.commit(GROUP_ID, replayed_batch)

assert [m.offset for m in first_batch] == [m.offset for m in replayed_batch]
assert replay_write["inserted"] == 0
first_write, replay_write, len(sink)


({'inserted': 11, 'duplicates': 1}, {'inserted': 0, 'duplicates': 12}, 11)

In [5]:
# Drain and commit the remaining topic.
while True:
    batch = broker.poll(GROUP_ID, max_records=10)
    if not batch:
        break
    write_idempotently(batch)
    broker.commit(GROUP_ID, batch)

assert len(sink) == len({event["event_id"] for event in source_events})

# Full replay: offsets go back to zero; the sink stays correct.
count_before_replay = len(sink)
broker.reset_group(GROUP_ID)
replay_duplicates = 0
while True:
    batch = broker.poll(GROUP_ID, max_records=25)
    if not batch:
        break
    replay_duplicates += write_idempotently(batch)["duplicates"]
    broker.commit(GROUP_ID, batch)

assert len(sink) == count_before_replay
print({"unique_sink_rows": len(sink), "duplicates_observed_during_replay": replay_duplicates})


{'unique_sink_rows': 45, 'duplicates_observed_during_replay': 47}


## Event time and late data

Processing time tells us when the consumer saw an event. Event time tells us when
the business action happened. A watermark allows bounded waiting for late data and
limits state. Events later than the allowed lateness need an explicit policy:
update a prior window, route to a late-data table, or reject with monitoring.


In [6]:
allowed_lateness = timedelta(minutes=3)
# Use producer arrival order. Iterating partition logs would group records by
# partition and would not reconstruct the original cross-partition arrival order.
arrival_order = source_events
max_event_time = None
late_events = []
for event in arrival_order:
    event_time = datetime.fromisoformat(event["event_time"])
    if max_event_time is not None and event_time < max_event_time - allowed_lateness:
        late_events.append(event)
    max_event_time = max(filter(None, [max_event_time, event_time]))

print("Late events beyond watermark:", len(late_events))
assert len(late_events) == 1
late_events[:2]


Late events beyond watermark: 1


[{'event_id': 'status-0040',
  'order_id': 'ord_000007',
  'status': 'packed',
  'event_time': '2026-01-05T00:01:00+00:00',
  'produced_at': '2026-09-04T19:12:30.963663+00:00'}]

## Optional: run against real Apache Kafka

From the project directory:

```bash
docker compose -f docker-compose.kafka.yml up -d
python -m pip install -r requirements-kafka.txt
```

Set `RUN_REAL_KAFKA = True` below. The code creates a three-partition topic,
publishes the same events with `order_id` keys, and consumes with manual commit.
Stop the broker later with:

```bash
docker compose -f docker-compose.kafka.yml down
```


In [7]:
RUN_REAL_KAFKA = False

if RUN_REAL_KAFKA:
    from kafka import KafkaAdminClient, KafkaConsumer, KafkaProducer
    from kafka.admin import NewTopic
    from kafka.errors import TopicAlreadyExistsError

    bootstrap = "localhost:9092"
    topic = "order-status"
    admin = KafkaAdminClient(bootstrap_servers=bootstrap, client_id="capstone-admin")
    try:
        admin.create_topics([NewTopic(topic, num_partitions=3, replication_factor=1)])
    except TopicAlreadyExistsError:
        pass
    finally:
        admin.close()

    producer = KafkaProducer(
        bootstrap_servers=bootstrap,
        key_serializer=lambda value: value.encode("utf-8"),
        value_serializer=lambda value: json.dumps(value).encode("utf-8"),
        acks="all",
        retries=3,
    )
    futures = [producer.send(topic, key=e["order_id"], value=e) for e in source_events]
    for future in futures:
        metadata = future.get(timeout=10)
        print(metadata.topic, metadata.partition, metadata.offset)
    producer.flush()
    producer.close()

    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=bootstrap,
        group_id="notebook-real-consumer-v1",
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        value_deserializer=lambda value: json.loads(value.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    real_sink = {}
    for message in consumer:
        real_sink[message.value["event_id"]] = {
            **message.value,
            "kafka_partition": message.partition,
            "kafka_offset": message.offset,
        }
    consumer.commit()
    consumer.close()
    print(f"Real Kafka sink rows: {len(real_sink)}")
else:
    print("Simulation mode complete. Set RUN_REAL_KAFKA=True after starting Docker.")


Simulation mode complete. Set RUN_REAL_KAFKA=True after starting Docker.


## Production boundaries

- A Kafka partition is an ordered log, but order is guaranteed only within that
  partition.
- Consumer-group members divide partitions; adding consumers beyond the partition
  count does not increase parallelism for that topic.
- Commit after durable output. A crash before commit causes replay; a commit before
  output can lose data.
- Idempotency can use a unique event key, database `MERGE`, transactional sink, or
  another deduplication state strategy.
- Exactly-once claims must name their boundary. End-to-end behavior includes the
  source, broker, processor, and sink.

## Your turn

1. Commit before writing, inject a crash, and show the lost record.
2. Use `customer_id` instead of `order_id` as the key and discuss ordering impact.
3. Start a second real consumer in the same group and inspect partition assignment.
4. Reset offsets and prove the real sink remains idempotent.
5. Design a dead-letter topic for records that fail schema validation.

### Final interview story

Connect the entire capstone: resilient API extraction, immutable Bronze data,
Spark validation and partitioned Silver output, ordered CDC, dimensional warehouse
models, Airflow recovery, and Kafka replay. State clearly which optional managed
services you actually used.
